# Data Preprocessing

This notebook implements the `DataPreprocessor` class used to clean raw text
corpora and validate SFT (Supervised Fine-Tuning) instruction datasets.

Key capabilities:

- **UTF-8 validation** — reads files as raw bytes and decodes strictly, reporting
  the byte offset of the first invalid sequence on failure.
- **HTML / control character removal** — strips `<tags>` and C0/C1 control codes
  while preserving common whitespace (`\t`, `\n`, `\r`).
- **Arabic normalization** — normalizes alef variants (آ أ إ → ا) via a
  configurable flag.
- **Corpus statistics & validation** — counts lines, bytes, and characters;
  checks minimum-size thresholds.
- **SFT data parsing** — reads JSONL or JSON-list files, validates
  `instruction` / `output` fields, and counts unique pairs.

## Imports and Constants

In [ ]:
import json
import logging
import re
import unicodedata

logger = logging.getLogger(__name__)

# Arabic alef variants to normalize → bare alef (U+0627)
_ALEF_VARIANTS = re.compile("[\u0622\u0623\u0625]")  # آ أ إ → ا

# HTML tag pattern
_HTML_TAG = re.compile(r"<[^>]+>")

# Control characters (C0/C1) except common whitespace (\t \n \r)
_CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")

## Private Helpers

Low-level I/O and parsing utilities used by the `DataPreprocessor` methods.

| Helper | Purpose |
|---|---|
| `_read_file_bytes` | Read a file as raw bytes (for strict UTF-8 decoding) |
| `_read_file_text` | Read a file as UTF-8 text (for JSON/JSONL parsing) |
| `_decode_utf8_strict` | Decode bytes → str, re-raising `UnicodeDecodeError` with byte offset |
| `_parse_json_or_jsonl` | Try JSON list first, fall back to line-delimited JSONL |

In [ ]:
def _read_file_bytes(filepath: str) -> bytes:
    """Read file as raw bytes."""
    with open(filepath, "rb") as f:
        return f.read()


def _read_file_text(filepath: str) -> str:
    """Read file as UTF-8 text."""
    with open(filepath, "r", encoding="utf-8") as f:
        return f.read()


def _decode_utf8_strict(data: bytes) -> str:
    """Decode bytes as UTF-8, raising UnicodeDecodeError with byte offset."""
    try:
        return data.decode("utf-8")
    except UnicodeDecodeError as e:
        raise UnicodeDecodeError(
            e.encoding, e.object, e.start, e.end, e.reason
        ) from None


def _parse_json_or_jsonl(text: str) -> list:
    """Parse text as either a JSON list or line-delimited JSONL.

    Tries JSON list first; if that fails or doesn't yield a list,
    falls back to JSONL (one JSON object per line).
    """
    stripped = text.strip()

    # Try JSON list first
    if stripped.startswith("["):
        parsed = json.loads(stripped)
        if isinstance(parsed, list):
            return parsed

    # Fall back to JSONL
    records: list = []
    for line_num, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise json.JSONDecodeError(
                f"Invalid JSON at line {line_num}: {e.msg}",
                e.doc,
                e.pos,
            ) from None
    return records

## DataPreprocessor Class

The main class that orchestrates all preprocessing. Instantiate with
optional flags to control Arabic normalization behaviour.

In [ ]:
class DataPreprocessor:
    """Clean and validate raw text corpora and SFT instruction datasets."""

    def __init__(
        self, normalize_arabic: bool = True, alef_normalization: bool = True
    ):
        """Configure preprocessing strategies.

        Args:
            normalize_arabic: Whether to apply Arabic text normalization.
            alef_normalization: Whether to normalize alef variants to bare alef.
        """
        self.normalize_arabic = normalize_arabic
        self.alef_normalization = alef_normalization

### `clean_text` — File Cleaning Pipeline

Reads a UTF-8 file and applies three cleaning steps in order:

1. **Remove HTML tags** — strips anything matching `<...>`.
2. **Remove control characters** — strips C0/C1 codes (`\x00`–`\x08`,
   `\x0b`, `\x0c`, `\x0e`–`\x1f`, `\x7f`–`\x9f`) while keeping
   `\t`, `\n`, and `\r`.
3. **Normalize Arabic alef** — replaces آ أ إ with bare ا when enabled.

Raises `UnicodeDecodeError` (with byte offset) if the file is not valid UTF-8,
and `FileNotFoundError` if the path doesn't exist.

In [ ]:
def clean_text(self, filepath: str) -> str:
    """Read a UTF-8 file, remove HTML/control chars, normalize Arabic."""
    raw_bytes = _read_file_bytes(filepath)
    text = _decode_utf8_strict(raw_bytes)

    # Remove HTML tags
    text = _HTML_TAG.sub("", text)

    # Remove control characters (keep \t, \n, \r)
    text = _CONTROL_CHARS.sub("", text)

    # Arabic normalization
    if self.normalize_arabic and self.alef_normalization:
        text = _ALEF_VARIANTS.sub("\u0627", text)

    return text


# Attach to the class
DataPreprocessor.clean_text = clean_text

### `get_corpus_stats` — Corpus Statistics

Returns a dict with three metrics:

| Key | Description |
|---|---|
| `line_count` | Number of lines (accounts for trailing newline) |
| `byte_size` | UTF-8 encoded size in bytes |
| `char_count` | Number of Unicode characters |

In [ ]:
def get_corpus_stats(self, text: str) -> dict:
    """Return corpus statistics."""
    return {
        "line_count": text.count("\n") + (1 if text and not text.endswith("\n") else 0),
        "byte_size": len(text.encode("utf-8")),
        "char_count": len(text),
    }


DataPreprocessor.get_corpus_stats = get_corpus_stats

### `validate_corpus` — Minimum Size Check

Returns `True` if the corpus has **at least** `min_lines` lines **OR** at least
`min_bytes` bytes. Logs a warning when neither threshold is met.

Default thresholds: 1 000 lines / 2 MB.

In [ ]:
def validate_corpus(
    self, text: str, min_lines: int = 1000, min_bytes: int = 2_000_000
) -> bool:
    """Check corpus meets minimum size requirements."""
    stats = self.get_corpus_stats(text)
    meets = stats["line_count"] >= min_lines or stats["byte_size"] >= min_bytes
    if not meets:
        logger.warning(
            "Corpus below minimum size: %d lines (need %d), %d bytes (need %d)",
            stats["line_count"],
            min_lines,
            stats["byte_size"],
            min_bytes,
        )
    return meets


DataPreprocessor.validate_corpus = validate_corpus

### `parse_sft_data` — SFT Record Parsing

Reads a JSONL or JSON-list file and validates each record:

- Must be a `dict`.
- Must have a non-empty `"instruction"` string.
- Must have a non-empty `"output"` string.
- An optional `"input"` field is preserved if present.

Invalid records are skipped with a logged warning including the record index.

In [ ]:
def parse_sft_data(self, filepath: str) -> list[dict]:
    """Parse JSONL or JSON list file and validate SFT records."""
    raw_text = _read_file_text(filepath)
    records = _parse_json_or_jsonl(raw_text)

    valid: list[dict] = []
    for idx, record in enumerate(records):
        if not isinstance(record, dict):
            logger.warning("Record %d: not a dict, skipping", idx)
            continue

        instruction = record.get("instruction")
        output = record.get("output")

        if not instruction or not isinstance(instruction, str) or not instruction.strip():
            logger.warning("Record %d: missing or empty 'instruction' field, skipping", idx)
            continue

        if not output or not isinstance(output, str) or not output.strip():
            logger.warning("Record %d: missing or empty 'output' field, skipping", idx)
            continue

        entry: dict = {
            "instruction": instruction,
            "output": output,
        }
        if "input" in record:
            entry["input"] = record["input"]

        valid.append(entry)

    return valid


DataPreprocessor.parse_sft_data = parse_sft_data

### `validate_sft_dataset` — Unique Pair Threshold

Checks that the dataset contains at least `min_pairs` unique
`(instruction, output)` pairs. Duplicate pairs are collapsed before counting.

Default threshold: 200 unique pairs.

In [ ]:
def validate_sft_dataset(
    self, records: list[dict], min_pairs: int = 200
) -> bool:
    """Check SFT dataset has enough unique instruction-response pairs."""
    unique_pairs = {
        (r["instruction"], r["output"]) for r in records
    }
    meets = len(unique_pairs) >= min_pairs
    if not meets:
        logger.warning(
            "SFT dataset has %d unique pairs (need %d)",
            len(unique_pairs),
            min_pairs,
        )
    return meets


DataPreprocessor.validate_sft_dataset = validate_sft_dataset

---

## Example Usage

The cells below demonstrate each method using in-memory sample data.
We write temporary files so the file-based methods work without external data.

In [ ]:
import tempfile
import os

pp = DataPreprocessor(normalize_arabic=True, alef_normalization=True)

### Demo: `clean_text`

Write a sample file containing HTML tags, control characters, and Arabic
alef variants, then clean it.

In [ ]:
sample_raw = (
    "<html><body>\n"
    "Hello world!\x07\n"                       # \x07 = BEL control char
    "<p>أهلاً وسهلاً</p>\n"                    # Arabic with HTML
    "الإسلام والإيمان\n"                        # إ should normalize to ا
    "آمنة وأحمد\n"                              # آ and أ should normalize
)

with tempfile.NamedTemporaryFile(mode="wb", suffix=".txt", delete=False) as f:
    f.write(sample_raw.encode("utf-8"))
    tmp_path = f.name

cleaned = pp.clean_text(tmp_path)
os.unlink(tmp_path)

print("=== Cleaned text ===")
print(repr(cleaned))
print()
print(cleaned)

### Demo: `get_corpus_stats`

In [ ]:
stats = pp.get_corpus_stats(cleaned)
print("Corpus statistics:", stats)

### Demo: `validate_corpus`

Our tiny sample won't meet the default thresholds (1 000 lines / 2 MB),
so we use small thresholds to show a passing case.

In [ ]:
# Fails with default thresholds
print("Meets default thresholds?", pp.validate_corpus(cleaned))

# Passes with small thresholds
print("Meets min_lines=3?", pp.validate_corpus(cleaned, min_lines=3, min_bytes=0))

### Demo: `parse_sft_data` (JSONL format)

Write a small JSONL file with valid and invalid records, then parse it.

In [ ]:
sft_jsonl = """{"instruction": "Translate to Arabic", "output": "ترجم إلى العربية", "input": "Hello"}
{"instruction": "Summarize", "output": "لخّص النص التالي"}
{"instruction": "", "output": "missing instruction"}
{"instruction": "No output field"}
{"instruction": "اكتب قصة قصيرة", "output": "كان يا ما كان في قديم الزمان"}
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False, encoding="utf-8") as f:
    f.write(sft_jsonl)
    sft_path = f.name

records = pp.parse_sft_data(sft_path)
os.unlink(sft_path)

print(f"Valid records: {len(records)} out of 5")
for i, r in enumerate(records):
    print(f"  [{i}] instruction={r['instruction']!r}  output={r['output']!r}")

### Demo: `parse_sft_data` (JSON list format)

The parser also accepts a JSON array.

In [ ]:
sft_json_list = json.dumps([
    {"instruction": "ما عاصمة السعودية؟", "output": "الرياض"},
    {"instruction": "What is 2+2?", "output": "4"},
], ensure_ascii=False)

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False, encoding="utf-8") as f:
    f.write(sft_json_list)
    json_path = f.name

records_json = pp.parse_sft_data(json_path)
os.unlink(json_path)

print(f"Valid records from JSON list: {len(records_json)}")
for r in records_json:
    print(f"  instruction={r['instruction']!r}  output={r['output']!r}")

### Demo: `validate_sft_dataset`

In [ ]:
all_records = records + records_json
print(f"Total records: {len(all_records)}")
print("Meets min_pairs=200?", pp.validate_sft_dataset(all_records))
print("Meets min_pairs=3?", pp.validate_sft_dataset(all_records, min_pairs=3))